# E2b — Leaky vs Clean trên RT-DETR-L (kiểm chứng chéo model cho E2)
**Mục đích:** chứng minh mức thổi phồng mAP do leakage KHÔNG phụ thuộc model. E2 đã đo trên YOLO26s (CNN); E2b lặp đúng giao thức trên RT-DETR-L (Transformer). Nếu hai model cho mức thổi phồng tương đương → claim trung tâm của paper vững gấp đôi.

- **leaky**: split GỐC của tác giả (augment trước khi chia)
- **clean**: gom toàn bộ ảnh, chia lại 70/20/10 theo NHÓM ảnh gốc (group-aware, seed 42)
- Cùng epochs (15), cùng seed, cùng imgsz 640 như E2 — chỉ đổi model.

**Cách chạy:** GPU **T4 x2** · Internet **ON** · điền `ROBOFLOW_API_KEY` (key MỚI) · Save & Run All.
KHÔNG cần Add Input. Ước tính **~7-8h** (RT-DETR nặng hơn YOLO ~2.5x; QUICK_TEST=True: ~40 phút).
**Có checkpoint-resume 2 tầng:** theo từng lượt train (state.json) VÀ giữa chừng một lượt (last.pt).
Nếu bị cắt 12h: Save Version xong → mở Edit → Add Input → chọn output notebook này → Run All lại.

In [ ]:
# ==================== CONFIG (E2b) ====================
QUICK_TEST = False
EPOCHS   = 2 if QUICK_TEST else 15
FRACTION = 0.15 if QUICK_TEST else 1.0
SEED = 42
ROBOFLOW_API_KEY = ""   # <== ĐIỀN KEY MỚI TẠI ĐÂY (đừng public notebook khi còn key)

import os, random, shutil, glob, json, time, yaml
WORK = "/kaggle/working"; os.makedirs(f"{WORK}/results", exist_ok=True)

STATE_PATH = f"{WORK}/state.json"
def save_state(): json.dump(STATE, open(STATE_PATH, "w"), indent=1, default=str)

# RESUME tầng 1: nạp state.json từ output lần chạy trước của CHÍNH E2b
STATE = {}
for _st in glob.glob("/kaggle/input/**/state.json", recursive=True):
    try: _s = json.load(open(_st))
    except Exception: continue
    if any("_rtdetr_" in k for k in _s):                      # đúng chữ ký E2b
        # GUARD: từ chối state của lần chạy QUICK_TEST (mọi lượt final < 5 phút)
        mins = [v.get("min", 0) for v in _s.values() if isinstance(v, dict) and "min" in v]
        if not QUICK_TEST and mins and max(mins) < 5:
            print("BỎ QUA state quick-test:", _st); continue
        STATE = _s; shutil.copy(_st, STATE_PATH)
        print("ĐÃ NẠP CHECKPOINT E2b từ:", _st); break
if not STATE and os.path.exists(STATE_PATH):
    STATE = json.load(open(STATE_PATH))
def done(k): return k in STATE
def mark(k, v): STATE[k] = v; save_state()
assert ROBOFLOW_API_KEY, "Điền ROBOFLOW_API_KEY (key MỚI — key cũ đã lộ phải revoke)!"
print("[E2b] QUICK_TEST=", QUICK_TEST, "| lượt đã xong:", [k for k in STATE if "_rtdetr_" in k] or "(chưa có)")

In [ ]:
%pip install -q ultralytics roboflow
import torch
assert torch.cuda.is_available(), "Bật GPU T4!"
cap = torch.cuda.get_device_capability(0)
assert cap[0] >= 7, f"GPU sm_{cap[0]}{cap[1]} quá cũ — chọn T4 x2!"
print("GPU:", torch.cuda.get_device_name(0), "- OK")

## 1. Tải 2 dataset gốc (giữ nguyên split leaky của tác giả)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
DATASETS = {
    "B_testcasque": ("testcasque", "ppe-detection-qlq3d", 1),
    "C_mohamed":    ("mohamed-traore-2ekkp", "ppe-detection-l80fg", 5),
}
DS_DIR = {}
for name, (ws, proj, ver) in DATASETS.items():
    out = f"{WORK}/{name}"
    if not os.path.exists(f"{out}/data.yaml"):
        rf.workspace(ws).project(proj).version(ver).download("yolov8", location=out)
    DS_DIR[name] = out
    n = sum(len(glob.glob(f"{out}/{sp}/images/*")) for sp in ["train","valid","test"])
    print(f"{name}: {n} ảnh ->", out)

## 2. Dựng bản CLEAN — chia lại 70/20/10 theo nhóm ảnh gốc (giống hệt E2)

In [ ]:
def src_of(path):
    s = os.path.basename(path)
    return s.split(".rf.")[0] if ".rf." in s else os.path.splitext(s)[0]

def fix_yaml(root):
    y0 = yaml.safe_load(open(f"{root}/data.yaml"))
    names = y0["names"]
    y = {"path": root, "train": "train/images", "val": "valid/images", "test": "test/images",
         "names": names, "nc": len(names)}
    p = f"{root}/data_fixed.yaml"; yaml.safe_dump(y, open(p, "w")); return p

CLEAN_DIR, LEAKY_YAML, CLEAN_YAML = {}, {}, {}
for name, root in DS_DIR.items():
    items = []
    for sp in ["train", "valid", "test"]:
        for img in sorted(glob.glob(f"{root}/{sp}/images/*")):
            lbl = f"{root}/{sp}/labels/{os.path.splitext(os.path.basename(img))[0]}.txt"
            items.append((img, lbl if os.path.exists(lbl) else None, sp))
    groups = {}
    for it in items: groups.setdefault(src_of(it[0]), []).append(it)
    out_root = f"{WORK}/{name}_clean"
    if not os.path.exists(f"{out_root}/data.yaml"):
        for sp in ["train", "valid", "test"]:
            os.makedirs(f"{out_root}/{sp}/images", exist_ok=True)
            os.makedirs(f"{out_root}/{sp}/labels", exist_ok=True)
        keys = sorted(groups); random.Random(SEED).shuffle(keys)
        total = len(items); acc = 0
        for k in keys:
            sp = "train" if acc < 0.7 * total else ("valid" if acc < 0.9 * total else "test")
            for img, lbl, _ in groups[k]:
                shutil.copy(img, f"{out_root}/{sp}/images/")
                if lbl: shutil.copy(lbl, f"{out_root}/{sp}/labels/")
                acc += 1
        shutil.copy(f"{root}/data.yaml", f"{out_root}/data.yaml")
    CLEAN_DIR[name] = out_root
    LEAKY_YAML[name] = fix_yaml(root); CLEAN_YAML[name] = fix_yaml(out_root)
    for sp in ["train", "valid", "test"]:
        print(f"  {name}_clean/{sp}:", len(glob.glob(f"{out_root}/{sp}/images/*")), "ảnh")

## 3. Train RT-DETR-L: 4 lượt (B-leaky, B-clean, C-leaky, C-clean) — AdamW lr=1e-4 (theo HPO E1), resume giữa chừng bằng last.pt

In [ ]:
from ultralytics import RTDETR
import pandas as pd, gc
rows = []
for name in DS_DIR:
    for arm, dyaml in [("leaky", LEAKY_YAML[name]), ("clean", CLEAN_YAML[name])]:
        key = f"{name}_rtdetr_{arm}"
        if done(key):
            rows.append(STATE[key]); print("skip:", STATE[key]); continue
        t0 = time.time()
        last = f"{WORK}/runs/{key}/weights/last.pt"
        if os.path.exists(last):
            print("RESUME giữa chừng từ:", last)
            m = RTDETR(last)
            m.train(resume=True)
        else:
            m = RTDETR("rtdetr-l.pt")
            m.train(data=dyaml, epochs=EPOCHS, imgsz=640, batch=8, seed=SEED, fraction=FRACTION,
                    optimizer="AdamW", lr0=1e-4,
                    project=f"{WORK}/runs", name=key, exist_ok=True, plots=False, verbose=False)
        v = m.val(data=dyaml, split="val", verbose=False)
        t = m.val(data=dyaml, split="test", verbose=False)
        row = {"dataset": name, "model": "RT-DETR-L", "arm": arm, "epochs": EPOCHS,
               "mAP50_val": round(v.box.map50, 4), "mAP50_test": round(t.box.map50, 4),
               "mAP50_95_test": round(t.box.map, 4), "min": round((time.time() - t0) / 60, 1)}
        rows.append(row); mark(key, row); print(row)
        del m; gc.collect(); torch.cuda.empty_cache()

## 4. Bảng kết quả: mức thổi phồng theo model (đối chiếu YOLO26s từ E2)

In [ ]:
df = pd.DataFrame(rows)
piv = df.pivot(index="dataset", columns="arm", values="mAP50_test")
piv["inflation_mAP50"] = (piv["leaky"] - piv["clean"]).round(4)
piv95 = df.pivot(index="dataset", columns="arm", values="mAP50_95_test")
piv["inflation_mAP50_95"] = (piv95["leaky"] - piv95["clean"]).round(4)
display(df); display(piv)
# Đối chiếu E2 (YOLO26s) — số đã chốt trong paper
E2_YOLO = {"B_testcasque": 0.0192, "C_mohamed": 0.0406}
for ds, infl in E2_YOLO.items():
    if ds in piv.index:
        print(f"{ds}: inflation RT-DETR-L = {piv.loc[ds,'inflation_mAP50']:.4f} | YOLO26s (E2) = {infl:.4f}")
df.to_csv(f"{WORK}/results/e2b_runs.csv", index=False)
piv.to_csv(f"{WORK}/results/e2b_inflation.csv")
print("\n>>> KẾT LUẬN E2b: nếu inflation hai model cùng dấu và cùng bậc -> leakage là thuộc tính của DỮ LIỆU, không phải của model.")

In [ ]:
# Dọn output: XÓA toàn bộ dataset đã tải — output chỉ còn results/ + runs/ + state.json
for name in list(DS_DIR):
    shutil.rmtree(DS_DIR[name], ignore_errors=True)
    shutil.rmtree(f"{WORK}/{name}_clean", ignore_errors=True)
# runs/ giữ weights để resume; nếu muốn output gọn hẳn sau khi XONG CẢ 4 lượt, bỏ comment 2 dòng dưới:
# if all(done(f"{n}_rtdetr_{a}") for n in DATASETS for a in ["leaky","clean"]):
#     shutil.rmtree(f"{WORK}/runs", ignore_errors=True)
print("Đã dọn. Output = results/ + runs/ + state.json")